In [1]:
import sys
sys.path.append("..")

from pathlib import Path
from sentence_transformers import SentenceTransformer

from src.data import load_dataset
from src.embedder import record_to_text, get_embeddings, save_embeddings, load_embeddings, get_cache_path
from src.search import search_all_questions
from src.metrics import evaluate
from src.config import MODELS, TOP_K

corpus, questions, categories = load_dataset()
corpus_texts = [record_to_text(item) for item in corpus]

print(f"Корпус: {len(corpus)} фрагментов")
print(f"Вопросы: {len(questions)} вопросов")

Корпус: 200 фрагментов
Вопросы: 25 вопросов


In [2]:
model_name = MODELS[0]
cache_path = get_cache_path(model_name)

if Path(cache_path).exists():
    corpus_embedding = load_embeddings(cache_path)
    print("Загружена из кэша")
else:
    corpus_embedding = get_embeddings(corpus_texts, model_name)
    save_embeddings(cache_path, corpus_embedding)
    print("Вычислено и добавлено в кэш")

Загружена из кэша


In [3]:
model = SentenceTransformer(model_name)
search_outputs = search_all_questions(questions, model, corpus, corpus_embedding)

metrics = evaluate(search_outputs, k=TOP_K)
print(metrics)
print("Для модели:", model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'precision_at_3': 0.84, 'mrr': 0.56, 'num_questions': 25}
Для модели: paraphrase-multilingual-MiniLM-L12-v2
